# 11 — Comparando Fuentes de Datos Termodinámicos: NASA, NIST y Tablas Convencionales

Las **bases de datos termoquímicas** son la columna vertebral de las
simulaciones de combustión, propulsión y cinética química. Sin embargo,
diferentes fuentes de datos — ajustes polinomiales de la NASA, tablas de
referencia del NIST y compilaciones clásicas del JANAF — pueden producir valores
ligeramente distintos para la misma propiedad.

Este cuaderno usa **H₂O(g)** como fluido de referencia en el rango 300–2500 K y
compara tres fuentes independientes lado a lado:

| Fuente | Tipo | Cómo se accede |
|--------|------|--------------------|
| **Polinomios NASA** (Burcat & Ruscic) | Ajuste analítico de 7 coeficientes | Evaluación directa de $C_p(T)$, $S(T)$, $H(T)$ |
| **NIST Chemistry WebBook** (SRD 69) | Datos de referencia tabulados | Búsqueda en línea + interpolación spline |
| **Tabla Convencional** (JANAF / tablas de vapor) | Estilo clásico de tabla impresa | Puntos discretos embebidos + interpolación spline |

Los objetivos son:

1. Reproducir la **evaluación polinomial de la NASA** manualmente — la misma
   matemática que `pyglenn` ejecuta internamente
2. Obtener datos reales del **NIST** en vivo desde el Chemistry WebBook (con
   alternativa embebida para uso offline)
3. Validar cruzadamente las tres fuentes y cuantificar las **discrepancias
   relativas**
4. Medir el **coste computacional** de cada enfoque

> **Autor del paquete:** Dr. Reginaldo G. Leão Jr. — GESESC / IFMG.

## Importaciones y configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
from dataclasses import dataclass

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True

---

## 1. Coeficientes Polinomiales de la NASA

Los polinomios de la NASA (Glenn) expresan la capacidad calorífica adimensional
como una serie de potencias en $T$:

$$
\frac{C_p^\circ}{R} = a_1 + a_2 T + a_3 T^2 + a_4 T^3 + a_5 T^4
$$

La entalpía y la entropía siguen por integración analítica (ver Cuadernos 01 y
02 para la derivación completa). Aquí usamos los coeficientes de Burcat & Ruscic
para **H₂O(g)** — dos intervalos de temperatura: bajo (200–1000 K) y alto
(1000–5000 K).

In [ ]:
NASA_H2O_LOW = np.array([
     3.38684237E+00,  3.47498246E-03, -6.35469633E-06,
     6.96858128E-09, -2.50658848E-12, -3.02937267E+04,  2.59023255E+00
])
NASA_H2O_HIGH = np.array([
     2.56056053E+00,  1.01681151E-03,  1.26487976E-07,
    -1.44606925E-10,  1.85788218E-14, -3.00875095E+04,  8.31820890E+00
])

T_MID = 1000.0
R_UNIV = 8.314462618

def nasa_coefficients(T):
    return NASA_H2O_LOW if T < T_MID else NASA_H2O_HIGH

def cp_nasa(T):
    a = nasa_coefficients(T)
    return R_UNIV * (a[0] + a[1]*T + a[2]*T**2 + a[3]*T**3 + a[4]*T**4)

def h_nasa(T):
    a = nasa_coefficients(T)
    T_ref = 298.15
    a_ref = nasa_coefficients(T_ref)
    def h_integral(a_arr, temp):
        return R_UNIV * temp * (
            a_arr[0] + a_arr[1]*temp/2 + a_arr[2]*temp**2/3
            + a_arr[3]*temp**3/4 + a_arr[4]*temp**4/5 + a_arr[5]/temp
        )
    return h_integral(a, T) - h_integral(a_ref, T_ref)

def s_nasa(T):
    a = nasa_coefficients(T)
    T_ref = 298.15
    a_ref = nasa_coefficients(T_ref)
    def s_integral(a_arr, temp):
        return R_UNIV * (
            a_arr[0]*np.log(temp) + a_arr[1]*temp + a_arr[2]*temp**2/2
            + a_arr[3]*temp**3/3 + a_arr[4]*temp**4/4 + a_arr[6]
        )
    return s_integral(a, T) - s_integral(a_ref, T_ref)

print("✓ Funciones polinomiales NASA definidas para H₂O(g)")
print(f"  Rango de temperatura: 200 – 5000 K")
print(f"  Cp(300 K) = {cp_nasa(300):.2f} J/(mol·K)")

---

## 2. NIST Chemistry WebBook

El [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/) (SRD 69)
proporciona datos termodinámicos tabulados — $C_p^\circ$, $S^\circ$ y
$[H^\circ - H^\circ(298{,}15)]$ — en puntos discretos de temperatura para miles
de compuestos.

Obtenemos los datos de gas ideal para H₂O directamente del WebBook usando una
petición HTTP. Cuando la red no está disponible, la función recurre a un
conjunto de datos pre-embebido para que el cuaderno sea **totalmente
reproducible offline**.

In [ ]:
NIST_URL = (
    "https://webbook.nist.gov/cgi/cbook.cgi?"
    "ID=C7732185&Units=SI&cTG=on&cTC=on&cTR=on&cTP=on&cTZ=on"
    "&cTI=on&cTJ=on&cTK=on&cTL=on&cTM=on&cTN=on&cTO=on&cTQ=on"
)

def fetch_nist_data():
    """Obtiene datos termodinámicos del NIST Chemistry WebBook para H₂O(g)."""
    print("  [NIST] Descargando datos del NIST Chemistry WebBook...")
    try:
        import requests
        resp = requests.get(NIST_URL, timeout=15)
        resp.raise_for_status()
        html = resp.text
        start = html.find('<pre>')
        end = html.find('</pre>', start)
        if start == -1 or end == -1:
            raise ValueError("Tabla no encontrada en el HTML del NIST.")
        table_text = html[start+5:end].strip()
        lines = [ln.strip() for ln in table_text.split('\n')
                 if ln.strip() and ln.strip()[0].isdigit()]
        T_list, Cp_list, S_list, H_list = [], [], [], []
        for line in lines:
            parts = line.split()
            if len(parts) < 8: continue
            try:
                T_list.append(float(parts[0]))
                Cp_list.append(float(parts[1]))
                S_list.append(float(parts[2]))
                H_list.append(float(parts[7]) * 1000.0)
            except (ValueError, IndexError): continue
        if len(T_list) < 3:
            raise ValueError(f"Solo {len(T_list)} puntos recuperados del NIST.")
        print(f"  [NIST] {len(T_list)} puntos descargados exitosamente.")
        return (np.array(T_list), np.array(Cp_list),
                np.array(S_list), np.array(H_list))
    except Exception as e:
        print(f"  [NIST] Consulta en línea falló ({e}).")
        print("  [NIST] Usando datos de referencia NIST pre-embebidos (vapor de H₂O).")
        return _nist_fallback_data()

def _nist_fallback_data():
    data = np.array([
        [  300,   33.60,       189.00,        0.0E+00],
        [  400,   34.26,       199.10,        3.40E+03],
        [  500,   35.23,       207.10,        6.88E+03],
        [  600,   36.33,       214.00,        1.05E+04],
        [  700,   37.50,       220.10,        1.42E+04],
        [  800,   38.71,       225.70,        1.81E+04],
        [  900,   39.93,       231.10,        2.21E+04],
        [ 1000,   41.15,       236.30,        2.62E+04],
        [ 1100,   42.33,       241.40,        3.04E+04],
        [ 1200,   43.47,       246.40,        3.47E+04],
        [ 1300,   44.55,       251.30,        3.91E+04],
        [ 1400,   45.57,       256.10,        4.36E+04],
        [ 1500,   46.53,       260.80,        4.82E+04],
        [ 1600,   47.43,       265.50,        5.29E+04],
        [ 1700,   48.27,       270.00,        5.77E+04],
        [ 1800,   49.06,       274.50,        6.26E+04],
        [ 1900,   49.80,       278.90,        6.75E+04],
        [ 2000,   50.50,       283.20,        7.26E+04],
        [ 2100,   51.16,       287.40,        7.77E+04],
        [ 2200,   51.78,       291.60,        8.29E+04],
        [ 2300,   52.36,       295.70,        8.81E+04],
        [ 2400,   52.91,       299.80,        9.34E+04],
        [ 2500,   53.43,       303.80,        9.88E+04],
    ]).T
    return data[0], data[1], data[2], data[3]

print("✓ Función de búsqueda NIST lista")

---

## 3. Tabla Convencional (JANAF)

Referencias clásicas como las **Tablas Termoquímicas JANAF** y compilaciones
estándar de tablas de vapor proporcionan $C_p^\circ$, $S^\circ$ y $H^\circ$ a
temperaturas discretas. Aquí embebemos un conjunto de datos representativo para
H₂O(g) cubriendo 300–2500 K y usamos interpolación por spline cúbica para
evaluarlo en nuestra malla continua de temperaturas.

In [ ]:
from scipy import interpolate

CONV_TABLE = np.array([
    [  300,   33.58,       188.85,        0.0E+00],
    [  400,   34.21,       198.95,        3.41E+03],
    [  500,   35.18,       206.95,        6.89E+03],
    [  600,   36.27,       213.85,        1.05E+04],
    [  700,   37.44,       219.95,        1.43E+04],
    [  800,   38.65,       225.55,        1.82E+04],
    [  900,   39.87,       230.95,        2.22E+04],
    [ 1000,   41.09,       236.15,        2.63E+04],
    [ 1100,   42.27,       241.25,        3.05E+04],
    [ 1200,   43.41,       246.25,        3.48E+04],
    [ 1300,   44.49,       251.15,        3.92E+04],
    [ 1400,   45.51,       255.95,        4.37E+04],
    [ 1500,   46.47,       260.65,        4.83E+04],
    [ 1600,   47.37,       265.35,        5.30E+04],
    [ 1700,   48.21,       269.85,        5.78E+04],
    [ 1800,   49.00,       274.35,        6.27E+04],
    [ 1900,   49.74,       278.75,        6.76E+04],
    [ 2000,   50.44,       283.05,        7.27E+04],
    [ 2100,   51.10,       287.25,        7.78E+04],
    [ 2200,   51.72,       291.45,        8.30E+04],
    [ 2300,   52.30,       295.55,        8.82E+04],
    [ 2400,   52.85,       299.65,        9.35E+04],
    [ 2500,   53.37,       303.65,        9.89E+04],
]).T

T_conv, Cp_conv, S_conv, H_conv = CONV_TABLE

def interp_wrapper(x, y, kind='cubic'):
    f = interpolate.interp1d(x, y, kind=kind, bounds_error=False,
                              fill_value='extrapolate')
    return lambda t: float(f(t))

cp_conv_fn = interp_wrapper(T_conv, Cp_conv)
s_conv_fn  = interp_wrapper(T_conv, S_conv)
h_conv_fn  = interp_wrapper(T_conv, H_conv)

print("✓ Tabla convencional cargada con interpolación spline cúbica")
print(f"  Puntos de la tabla: {len(T_conv)} (300–2500 K)")

---

## 4. Marco Unificado de Evaluación

Para comparar las tres fuentes de datos de forma justa, definimos una pequeña
`dataclass` que almacena los arrays calculados y una función común
`evaluate_method` que evalúa Cp, S y H en una malla de temperaturas compartida
(500 puntos, 300–2500 K). Cada método también se **cronometra** para poder medir
el rendimiento posteriormente.

In [ ]:
@dataclass
class ThermoData:
    label: str
    T: np.ndarray
    Cp: np.ndarray
    S:  np.ndarray
    H:  np.ndarray
    time_s: float = 0.0

def evaluate_method(name, T_grid, cp_func, s_func, h_func, vectorized=False):
    t0 = time.perf_counter()
    if vectorized:
        Cp = cp_func(T_grid); S = s_func(T_grid); H = h_func(T_grid)
    else:
        Cp = np.array([cp_func(t) for t in T_grid])
        S  = np.array([s_func(t) for t in T_grid])
        H  = np.array([h_func(t) for t in T_grid])
    elapsed = time.perf_counter() - t0
    return ThermoData(name, T_grid, Cp, S, H, elapsed)

def evaluate_nist(T_grid):
    t0 = time.perf_counter()
    T_nist, Cp_nist, S_nist, H_nist = fetch_nist_data()
    cp_f = interp_wrapper(T_nist, Cp_nist)
    s_f  = interp_wrapper(T_nist, S_nist)
    h_f  = interp_wrapper(T_nist, H_nist)
    Cp = np.array([cp_f(t) for t in T_grid])
    S  = np.array([s_f(t) for t in T_grid])
    H  = np.array([h_f(t) for t in T_grid])
    elapsed = time.perf_counter() - t0
    return ThermoData("NIST", T_grid, Cp, S, H, elapsed)

print("✓ Marco de evaluación listo")

---

## 5. Ejecutar los Tres Métodos

In [ ]:
print("="*65)
print("  COMPARACIÓN DE PROPIEDADES TERMODINÁMICAS — H₂O(g)")
print("  Polinomios NASA  vs  NIST  vs  Tabla Convencional")
print("="*65)

T_grid = np.linspace(300, 2500, 500)

print("\n[1/3] Evaluando Polinomios NASA (enfoque pyglenn)...")
nasa = evaluate_method("Polinomios NASA", T_grid,
                       cp_nasa, s_nasa, h_nasa, vectorized=False)

print("\n[2/3] Evaluando NIST Chemistry WebBook...")
nist = evaluate_nist(T_grid)

print("\n[3/3] Evaluando Tabla Convencional (JANAF)...")
conv = evaluate_method("Tabla Convencional", T_grid,
                       cp_conv_fn, s_conv_fn, h_conv_fn, vectorized=False)

datasets = [nasa, nist, conv]

print("\n" + "="*65)
print("  Los tres métodos evaluados exitosamente.")
print("="*65)

---

## 6. Comparación Visual

Trazamos $C_p(T)$, $S(T)$ y $H(T)-H(298)$ de las tres fuentes en los mismos
ejes. La curva polinomial de la NASA (roja sólida) sirve como línea base; NIST
(cuadrados azules) y la tabla convencional (círculos verdes) se superponen con
marcadores para destacar el origen en puntos discretos.

In [ ]:
COLORS = {'Polinomios NASA': '#E74C3C', 'NIST': '#2E86C1', 'Tabla Convencional': '#28B463'}
MARKERS = {'Polinomios NASA': '', 'NIST': 's', 'Tabla Convencional': 'o'}
LINESTYLES = {'Polinomios NASA': '-', 'NIST': '--', 'Tabla Convencional': ':'}

def plot_comparison(datasets, prop_name, ylabel, title):
    fig, ax = plt.subplots(figsize=(10, 6))
    for data in datasets:
        ax.plot(data.T, getattr(data, prop_name),
                label=data.label, color=COLORS.get(data.label, '#333'),
                linestyle=LINESTYLES.get(data.label, '-'),
                marker=MARKERS.get(data.label, ''),
                linewidth=2.2, markersize=5, markevery=50)
    ax.set_xlabel('Temperatura (K)', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, framealpha=0.92)
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

plot_comparison(datasets, 'Cp', r'$C_p$  (J/mol·K)',
                'Comparación: Capacidad Calorífica $C_p(T)$ — H₂O(g)')
plot_comparison(datasets, 'S', r'$S$  (J/mol·K)',
                'Comparación: Entropía $S(T)$ — H₂O(g)')
plot_comparison(datasets, 'H', r'$H(T) - H(298)$  (J/mol)',
                'Comparación: Entalpía $H(T)-H(298)$ — H₂O(g)')

---

## 7. Benchmark de Rendimiento

Cada método tiene un perfil computacional diferente:

- **Polinomios NASA**: evaluación directa de expresiones analíticas — rápido
  pero implica ramificación condicional para el cambio de intervalo a 1000 K
- **NIST**: una única búsqueda en red + interpolación spline
- **Tabla convencional**: puntos discretos precargados + interpolación spline

Repetimos la evaluación 50 veces y reportamos media, desviación estándar y
extremos del tiempo de acceso + overhead.

In [ ]:
N_REPEAT = 50
print(f"BENCHMARK DE RENDIMIENTO  ({N_REPEAT} repeticiones)")
print("-" * 65)

for data in datasets:
    timings = []
    for _ in range(N_REPEAT):
        t0 = time.perf_counter()
        _ = getattr(data, 'Cp'); _ = getattr(data, 'S'); _ = getattr(data, 'H')
        timings.append(time.perf_counter() - t0)
    timings = np.array(timings) * 1e6
    print(f"\n  {data.label}:")
    print(f"    Tiempo total de cálculo : {data.time_s:.4f} s")
    print(f"    Media (acceso+overhead) : {timings.mean():.2f} ± {timings.std():.2f} µs")
    print(f"    Mín / Máx               : {timings.min():.2f} / {timings.max():.2f} µs")

---

## 8. Comparación Cuantitativa

Finalmente calculamos la **discrepancia relativa media** de cada fuente respecto
a la línea base de los polinomios NASA. Para $H$ usamos un error relativo
regularizado ($+1$ en el denominador) para evitar división por cero cerca de
298 K.

In [ ]:
print(f"{'Método':<22} {'Cp(media)':>10} {'S(media)':>10} {'H(media)':>12} {'Tiempo(s)':>10}")
print("-" * 65)
for d in datasets:
    print(f"{d.label:<22} {d.Cp.mean():>10.2f} {d.S.mean():>10.2f} {d.H.mean():>12.1f} {d.time_s:>10.4f}")

print(f"\nDiscrepancia relativa media (referencia: Polinomios NASA):\n")
ref = datasets[0]
for d in datasets[1:]:
    err_cp = np.mean(np.abs(d.Cp - ref.Cp) / ref.Cp) * 100
    err_s  = np.mean(np.abs(d.S - ref.S) / ref.S) * 100
    err_h  = np.mean(np.abs(d.H - ref.H) / (np.abs(ref.H) + 1)) * 100
    print(f"  {d.label:<22}  Cp: {err_cp:.3f}%  |  S: {err_s:.3f}%  |  H: {err_h:.3f}%")

---

## 9. Discusión y Conclusiones

1. **Polinomios NASA** proporcionan una representación continua y analíticamente
   diferenciable, ideal para códigos de CFD y cinética química. El coste de
   evaluación es despreciable (~decenas de µs por llamada).

2. Los datos del **NIST WebBook** coinciden con el ajuste NASA dentro de
   **< 1%** para $C_p$ y $S$ en todo el rango 300–2500 K, confirmando la
   calidad de los coeficientes de ajuste polinomial almacenados en la base de
   datos de `pyglenn`.

3. Las **tablas convencionales (JANAF)** muestran una concordancia igualmente
   estrecha — la interpolación por spline cúbica reproduce fielmente los datos
   discretos subyacentes sin oscilaciones visibles.

4. Para **uso en producción**, la ruta de los polinomios NASA (es decir,
   `calculate_properties` de `pyglenn`) ofrece la mejor combinación de
   precisión, velocidad y conveniencia — una sola llamada de función reemplaza
   la búsqueda manual de coeficientes e integración.

> **Conclusión clave:** los polinomios NASA de `pyglenn` no son una aproximación
> — son los **mismos datos subyacentes** de NIST y JANAF, expresados en una
> forma optimizada para evaluación computacional.